<a href="https://colab.research.google.com/github/dev0-hub/CosPackQRCode/blob/main/Finger_counting_with_hand_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q mediapipe opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 6.3 MB/s eta 0:00:00


In [21]:
import cv2
import numpy as np
import urllib.request
import time
import io
from base64 import b64decode, b64encode
from PIL import Image
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

In [22]:
model_url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"
urllib.request.urlretrieve(model_url, "hand_landmarker.task")
print("Model downloaded.")

Model downloaded.


In [23]:
def video_stream():
    js = Javascript('''
    var video; var div = null; var stream; var captureCanvas;
    var imgElement; var labelElement;
    var pendingResolve = null; var shutdown = false;

    function removeDom() {
      stream.getVideoTracks()[0].stop();
      video.remove(); div.remove();
      video = null; div = null; stream = null;
      imgElement = null; captureCanvas = null; labelElement = null;
    }

    function onAnimationFrame() {
      if (!shutdown) { window.requestAnimationFrame(onAnimationFrame); }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8);
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }

    async function createDom() {
      if (div !== null) { return stream; }
      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '640px';
      document.body.appendChild(div);

      const modelOut = document.createElement('div');
      modelOut.innerHTML = "<span>Status:</span>";
      labelElement = document.createElement('span');
      labelElement.innerText = 'No data';
      labelElement.style.fontWeight = 'bold';
      modelOut.appendChild(labelElement);
      div.appendChild(modelOut);

      video = document.createElement('video');
      video.style.display = 'block';
      video.width = 640;
      video.setAttribute('playsinline', '');
      video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia({video: true});
      div.appendChild(video);

      imgElement = document.createElement('img');
      imgElement.style.position = 'absolute';
      imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; };
      div.appendChild(imgElement);

      const instruction = document.createElement('div');
      instruction.innerHTML = '<span style="color:red;font-weight:bold;">Click video to stop</span>';
      div.appendChild(instruction);
      instruction.onclick = () => { shutdown = true; };

      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640;
      captureCanvas.height = 480;
      window.requestAnimationFrame(onAnimationFrame);
      return stream;
    }

    async function stream_frame(label, imgData) {
      if (shutdown) { removeDom(); shutdown = false; return ''; }
      stream = await createDom();
      if (label != "") { labelElement.innerHTML = label; }
      if (imgData != "") {
        var r = video.getClientRects()[0];
        imgElement.style.top = r.top + "px";
        imgElement.style.left = r.left + "px";
        imgElement.style.width = r.width + "px";
        imgElement.style.height = r.height + "px";
        imgElement.src = imgData;
      }
      var result = await new Promise((resolve) => { pendingResolve = resolve; });
      shutdown = false;
      return result;
    }
    ''')
    display(js)

def video_frame(label='', bbox=''):
    return eval_js('stream_frame("{}", "{}")'.format(label, bbox))

def wait_for_stream_ready(timeout=10):
    start = time.time()
    while time.time() - start < timeout:
        if eval_js('typeof stream_frame === "function"'):
            return True
        time.sleep(0.3)
    return False

In [11]:
base_options = mp_python.BaseOptions(model_asset_path='hand_landmarker.task')
options = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.6,
    min_hand_presence_confidence=0.6,
    min_tracking_confidence=0.6
)
detector = mp_vision.HandLandmarker.create_from_options(options)
print("Detector ready.")

Detector ready.


In [24]:
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (5,9),(9,10),(10,11),(11,12),
    (9,13),(13,14),(14,15),(15,16),
    (13,17),(17,18),(18,19),(19,20),
    (0,17)
]
FINGER_TIPS = [4, 8, 12, 16, 20]
FINGER_PIPS = [3, 6, 10, 14, 18]

def count_fingers(landmarks, handedness_label):
    fingers_up = []
    if handedness_label == "Right":
        fingers_up.append(1 if landmarks[4].x > landmarks[3].x else 0)
    else:
        fingers_up.append(1 if landmarks[4].x < landmarks[3].x else 0)
    for tip, pip in zip(FINGER_TIPS[1:], FINGER_PIPS[1:]):
        fingers_up.append(1 if landmarks[tip].y < landmarks[pip].y else 0)
    return sum(fingers_up)

def draw_landmarks_overlay(overlay, landmarks):
    h, w, _ = overlay.shape
    points = [(int(lm.x * w), int(lm.y * h)) for lm in landmarks]
    for a, b in HAND_CONNECTIONS:
        cv2.line(overlay, points[a], points[b], (0, 255, 0, 255), 2)
    for p in points:
        cv2.circle(overlay, p, 4, (0, 0, 255, 255), -1)

def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    return cv2.imdecode(jpg_as_np, flags=1)

def overlay_to_bytes(overlay_array):
    img = Image.fromarray(overlay_array, 'RGBA')
    buf = io.BytesIO()
    img.save(buf, format='png')
    return 'data:image/png;base64,' + b64encode(buf.getvalue()).decode('utf-8')

In [ ]:
video_stream()
if not wait_for_stream_ready():
    raise RuntimeError("Script didn't load in time — re-run this cell.")
print("Ready. Click the video anytime to stop.")

label_html = 'Tracking...'
bbox = ''

try:
    while True:
        js_reply = video_frame(label_html, bbox)
        if not js_reply:
            break

        frame = js_to_image(js_reply)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        result = detector.detect(mp_image)

        overlay = np.zeros((480, 640, 4), dtype=np.uint8)
        if result.hand_landmarks:
            total = 0
            for landmarks, handedness in zip(result.hand_landmarks, result.handedness):
                draw_landmarks_overlay(overlay, landmarks)
                total += count_fingers(landmarks, handedness[0].category_name)
            label_html = f'Fingers: {total}'
        else:
            label_html = 'No hand detected'

        bbox = overlay_to_bytes(overlay)

except KeyboardInterrupt:
    print("Stopped.")

<IPython.core.display.Javascript object>

Ready. Click the video anytime to stop.


/tmp/ipykernel_1742/4191167137.py:36: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(overlay_array, 'RGBA')
